# 01_Data_Understanding

**Phase 2, Step 2.1:** Load and inspect all 6 raw datasets from the PM Surya Ghar solar subsidy scheme.

This notebook:
- Loads raw CSVs from `raw_data/`
- Inspects schemas, row counts, data types
- Checks for missing values, anomalies, data quality issues
- Creates data dictionary with summary statistics
- Validates join keys and hierarchies

**Outputs:** Data understanding summary for Step 2.2 (cleaning)

In [1]:
import sys
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Change to project root directory
project_root = Path.cwd().parent
os.chdir(project_root)
sys.path.insert(0, str(project_root))

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print(f"Project root: {project_root}")
print(f"Working directory: {os.getcwd()}")
print(f"Raw data path: {project_root / 'raw_data'}")

Project root: c:\Users\Rahul\Desktop\Data Analysis\Suryaghar_Data Analysis Project
Working directory: c:\Users\Rahul\Desktop\Data Analysis\Suryaghar_Data Analysis Project
Raw data path: c:\Users\Rahul\Desktop\Data Analysis\Suryaghar_Data Analysis Project\raw_data


## 1. Load All Raw Datasets

Load all 6 CSV files from `raw_data/` using the data loading module.

In [4]:
import importlib.util

# Load the data_loader module from scripts folder
spec = importlib.util.spec_from_file_location(
    "data_loader",
    str(Path.cwd() / "scripts" / "00_data_loader.py")
)
data_loader = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_loader)

# Load all raw datasets
datasets = data_loader.load_all_datasets()

print("✓ Successfully loaded all datasets:")
for name, df in datasets.items():
    print(f"  - {name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

✓ Successfully loaded all datasets:
  - datewise: 795 rows × 20 columns
  - state_master: 79 rows × 34 columns
  - district: 794 rows × 29 columns
  - discom_master: 87 rows × 27 columns
  - subsidy_status: 29 rows × 3 columns
  - vendor_selection: 90 rows × 8 columns


## 2. Dataset Schemas & Data Types

Inspect the schema, columns, and data types for each dataset.

In [5]:
for name, df in datasets.items():
    print(f"\n{'='*80}")
    print(f"Dataset: {name.upper()}")
    print(f"{'='*80}")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"\nData Types:")
    print(df.dtypes)
    print(f"\nFirst 3 Rows:")
    print(df.head(3))


Dataset: DATEWISE
Shape: 795 rows × 20 columns

Data Types:
#                    int64
rptdate                str
applications           str
residential            str
rwa                  int64
totalhouseholds        str
upto_10_kw             str
above_10_kw          int64
installations          str
inspection             str
subsidyredeemed        str
Unnamed: 11        float64
Unnamed: 12        float64
Unnamed: 13        float64
Unnamed: 14        float64
Unnamed: 15        float64
Unnamed: 16        float64
Unnamed: 17        float64
Unnamed: 18        float64
Unnamed: 19        float64
dtype: object

First 3 Rows:
   #     rptdate applications residential  rwa totalhouseholds upto_10_kw  \
0  1  09-02-2026            6           1    0               0          1   
1  2  08-02-2026        9,017       2,479    1              72      2,477   
2  3  07-02-2026       20,625       6,536   20             967      6,541   

   above_10_kw installations inspection subsidyredeemed  Unna

## 3. Missing Values & Data Quality

Check for NULL/NaN values and data quality issues in each dataset.

In [6]:
for name, df in datasets.items():
    print(f"\n{'='*80}")
    print(f"Missing Values: {name.upper()}")
    print(f"{'='*80}")
    missing = df.isnull().sum()
    missing_pct = (df.isnull().sum() / len(df)) * 100
    missing_df = pd.DataFrame({
        'Column': missing.index,
        'Missing_Count': missing.values,
        'Missing_Percent': missing_pct.values
    }).sort_values('Missing_Count', ascending=False)
    missing_df = missing_df[missing_df['Missing_Count'] > 0]
    
    if len(missing_df) == 0:
        print("✓ No missing values found")
    else:
        print(missing_df.to_string(index=False))


Missing Values: DATEWISE
     Column  Missing_Count  Missing_Percent
Unnamed: 16            795            100.0
Unnamed: 17            795            100.0
Unnamed: 18            795            100.0
Unnamed: 11            795            100.0
Unnamed: 19            795            100.0
Unnamed: 14            795            100.0
Unnamed: 13            795            100.0
Unnamed: 12            795            100.0
Unnamed: 15            795            100.0

Missing Values: STATE_MASTER
                                     Column  Missing_Count  Missing_Percent
                                          #             43        54.430380
                                      state             43        54.430380
                         application_status             42        53.164557
                            vendor_selected             42        53.164557
       vendor_selection_pending_gt_180_days             42        53.164557
                       feasibility_approved     

## 4. Duplicate Rows

Check for exact duplicate rows in each dataset.

In [7]:
for name, df in datasets.items():
    duplicates = df.duplicated().sum()
    print(f"{name}: {duplicates} duplicate rows")
    if duplicates > 0:
        print(f"  ⚠️  {duplicates} ({(duplicates/len(df)*100):.2f}%) duplicate rows found")

datewise: 0 duplicate rows
state_master: 41 duplicate rows
  ⚠️  41 (51.90%) duplicate rows found
district: 0 duplicate rows
discom_master: 1 duplicate rows
  ⚠️  1 (1.15%) duplicate rows found
subsidy_status: 1 duplicate rows
  ⚠️  1 (3.45%) duplicate rows found
vendor_selection: 5 duplicate rows
  ⚠️  5 (5.56%) duplicate rows found


## 5. Numeric Columns Analysis

Examine numeric columns for value ranges, anomalies, and Indian number formatting.

In [9]:
for name, df in datasets.items():
    print(f"\n{'='*80}")
    print(f"Numeric Columns: {name.upper()}")
    print(f"{'='*80}")
    
    # Find numeric columns
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    
    for col in numeric_cols:
        col_data = df[col]
        print(f"\n{col}: {col_data.dtype}")
        print(f"  Min: {col_data.min()}, Max: {col_data.max()}, Mean: {col_data.mean():.2f}")
    
    # Check for string columns with Indian numbers
    string_cols = df.select_dtypes(include=['object']).columns
    indian_number_cols = []
    for col in string_cols:
        col_data = df[col].dropna()
        if len(col_data) > 0:
            sample = col_data.head(3).tolist()
            if any(',' in str(v) for v in sample):
                indian_number_cols.append((col, sample))
    
    if indian_number_cols:
        print(f"\nColumns with Indian Number Formatting Detected:")
        for col, samples in indian_number_cols:
            print(f"  {col}: {samples}")


Numeric Columns: DATEWISE

#: int64
  Min: 1, Max: 795, Mean: 398.00

rwa: int64
  Min: 0, Max: 123, Mean: 16.04

above_10_kw: int64
  Min: 0, Max: 70, Mean: 13.27

Unnamed: 11: float64
  Min: nan, Max: nan, Mean: nan

Unnamed: 12: float64
  Min: nan, Max: nan, Mean: nan

Unnamed: 13: float64
  Min: nan, Max: nan, Mean: nan

Unnamed: 14: float64
  Min: nan, Max: nan, Mean: nan

Unnamed: 15: float64
  Min: nan, Max: nan, Mean: nan

Unnamed: 16: float64
  Min: nan, Max: nan, Mean: nan

Unnamed: 17: float64
  Min: nan, Max: nan, Mean: nan

Unnamed: 18: float64
  Min: nan, Max: nan, Mean: nan

Unnamed: 19: float64
  Min: nan, Max: nan, Mean: nan

Columns with Indian Number Formatting Detected:
  applications: ['6', '9,017', '20,625']
  residential: ['1', '2,479', '6,536']
  upto_10_kw: ['1', '2,477', '6,541']
  installations: ['1', '2,480', '6,556']
  inspection: ['37', '1,009', '5,965']
  subsidyredeemed: ['6', '2,660', '7,305']

Numeric Columns: STATE_MASTER

#: float64
  Min: 1.0, Max:

C:\Users\Rahul\AppData\Local\Temp\ipykernel_14696\2694806050.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = df.select_dtypes(include=['object']).columns
C:\Users\Rahul\AppData\Local\Temp\ipykernel_14696\2694806050.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_

## 6. Date Range Analysis

Check for date columns and examine time coverage in each dataset.

In [10]:
for name, df in datasets.items():
    print(f"\n{name.upper()}:")
    
    # Check common date column names
    date_cols = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
    
    if date_cols:
        for col in date_cols:
            try:
                # Try to parse as datetime
                date_data = pd.to_datetime(df[col], errors='coerce')
                if date_data.notna().sum() > 0:
                    print(f"  {col}: {date_data.min()} to {date_data.max()}")
                else:
                    print(f"  {col}: Unable to parse as datetime")
            except:
                print(f"  {col}: Error parsing dates")
    else:
        print("  No date columns found")


DATEWISE:
  rptdate: 2022-01-12 00:00:00 to 2026-12-01 00:00:00

STATE_MASTER:
  No date columns found

DISTRICT:
  No date columns found

DISCOM_MASTER:
  No date columns found

SUBSIDY_STATUS:
  No date columns found

VENDOR_SELECTION:
  No date columns found


## 7. Data Hierarchy Validation

Verify state-district-DISCOM relationships and hierarchy integrity.

In [11]:
# Check state_master hierarchy
print("STATE_MASTER Hierarchy:")
state_master = datasets.get('state_master', pd.DataFrame())
if not state_master.empty:
    if 'state' in state_master.columns:
        print(f"  Unique states: {state_master['state'].nunique()}")
    if 'state_code' in state_master.columns:
        print(f"  Unique state codes: {state_master['state_code'].nunique()}")

# Check district hierarchy
print("\nDISTRICT Hierarchy:")
district = datasets.get('district', pd.DataFrame())
if not district.empty:
    if 'state' in district.columns and 'district' in district.columns:
        print(f"  Unique states: {district['state'].nunique()}")
        print(f"  Unique districts: {district['district'].nunique()}")
        print(f"  Avg districts per state: {district.groupby('state')['district'].nunique().mean():.2f}")

# Check DISCOM hierarchy
print("\nDISCOM_MASTER Hierarchy:")
discom_master = datasets.get('discom_master', pd.DataFrame())
if not discom_master.empty:
    if 'state' in discom_master.columns and 'discom' in discom_master.columns:
        print(f"  Unique states: {discom_master['state'].nunique()}")
        print(f"  Unique DISCOMs: {discom_master['discom'].nunique()}")
        print(f"  Avg DISCOMs per state: {discom_master.groupby('state')['discom'].nunique().mean():.2f}")

STATE_MASTER Hierarchy:
  Unique states: 36

DISTRICT Hierarchy:
  Unique states: 36
  Unique districts: 789
  Avg districts per state: 22.00

DISCOM_MASTER Hierarchy:
  Unique states: 36
  Unique DISCOMs: 84
  Avg DISCOMs per state: 2.33


## 8. Comprehensive Data Dictionary

Create a master data dictionary documenting all datasets, columns, and their properties.

In [12]:
data_dictionary = []

for dataset_name, df in datasets.items():
    for col in df.columns:
        col_data = df[col]
        
        # Determine column type
        if col_data.dtype == 'object':
            col_type = 'String'
            # Check for Indian numbers
            sample = col_data.dropna().head(1).values
            if len(sample) > 0 and ',' in str(sample[0]):
                col_type = 'Indian_Number'
        elif col_data.dtype in ['int64', 'int32']:
            col_type = 'Integer'
        elif col_data.dtype in ['float64', 'float32']:
            col_type = 'Float'
        elif col_data.dtype == 'datetime64[ns]':
            col_type = 'DateTime'
        else:
            col_type = str(col_data.dtype)
        
        data_dictionary.append({
            'Dataset': dataset_name,
            'Column': col,
            'Type': col_type,
            'Non_Null': col_data.notna().sum(),
            'Null': col_data.isna().sum(),
            'Unique': col_data.nunique(),
            'Min': col_data.min() if col_data.dtype in ['int64', 'float64'] else None,
            'Max': col_data.max() if col_data.dtype in ['int64', 'float64'] else None
        })

data_dict_df = pd.DataFrame(data_dictionary)
print("DATA DICTIONARY")
print("="*100)
print(data_dict_df.to_string(index=False))

# Save to CSV for reference
# data_dict_df.to_csv(project_root / 'docs' / 'DATA_DICTIONARY.csv', index=False)

DATA DICTIONARY
         Dataset                                                Column    Type  Non_Null  Null  Unique  Min     Max
        datewise                                                     # Integer       795     0     795  1.0   795.0
        datewise                                               rptdate     str       795     0     795  NaN     NaN
        datewise                                          applications     str       795     0     718  NaN     NaN
        datewise                                           residential     str       795     0     683  NaN     NaN
        datewise                                                   rwa Integer       795     0      68  0.0   123.0
        datewise                                       totalhouseholds     str       795     0     539  NaN     NaN
        datewise                                            upto_10_kw     str       795     0     696  NaN     NaN
        datewise                                        

## 9. Summary & Key Findings

Document key observations about data quality, structure, and any anomalies discovered.

In [13]:
print("\n" + "="*80)
print("DATA UNDERSTANDING SUMMARY")
print("="*80)

total_rows = sum(df.shape[0] for df in datasets.values())
total_cols = sum(df.shape[1] for df in datasets.values())

print(f"\n📊 OVERALL STATISTICS:")
print(f"   Total Datasets: {len(datasets)}")
print(f"   Total Rows Across All Datasets: {total_rows:,}")
print(f"   Total Columns Across All Datasets: {total_cols}")

print(f"\n📋 DATASET BREAKDOWN:")
for name, df in datasets.items():
    print(f"   {name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

print(f"\n✓ Data Understanding Complete!")
print(f"   Next Step: Run 01_data_cleaning.py to clean and standardize data")


DATA UNDERSTANDING SUMMARY

📊 OVERALL STATISTICS:
   Total Datasets: 6
   Total Rows Across All Datasets: 1,874
   Total Columns Across All Datasets: 121

📋 DATASET BREAKDOWN:
   datewise: 795 rows × 20 columns
   state_master: 79 rows × 34 columns
   district: 794 rows × 29 columns
   discom_master: 87 rows × 27 columns
   subsidy_status: 29 rows × 3 columns
   vendor_selection: 90 rows × 8 columns

✓ Data Understanding Complete!
   Next Step: Run 01_data_cleaning.py to clean and standardize data
